# 01 - Data Understanding and Preprocessing

## Objective

This notebook performs the initial understanding, validation and preprocessing of the industrial dataset used for the Missed Delivery prediction project.

The dataset combines delivery outcomes with operational indicators related to production, demand, stock and planning compliance.

The main objectives are:

- Load and inspect the raw analytical dataset.
- Understand the dataset granularity and variable types.
- Convert temporal and numerical variables into appropriate data types.
- Validate missing values and duplicated observations.
- Analyse the historical coverage of the planning groups.
- Validate operational indicators and their underlying variables.
- Preserve operationally meaningful negative values.
- Define the requested delivery quantity.
- Define the binary Missed Delivery target.
- Quantify both Missed Delivery frequency and lost-volume impact.
- Export a clean dataset for subsequent Exploratory Data Analysis.

No temporal lag features or Machine Learning transformations are generated in this notebook.

## 1. Libraries

Only the libraries required for data manipulation and initial validation are imported at this stage.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

## 2. Load Raw Dataset

The raw analytical dataset was extracted from the industrial data platform through SQL.

Each record combines delivery outcomes with operational performance indicators associated with the corresponding planning group and date.

The original file is kept unchanged in the `data/raw` directory to ensure reproducibility.

In [2]:
file_path = "../../data/raw/dataset_raw.csv"
df = pd.read_csv(file_path, sep = ";")
df.head()

,Fecha,Grupo_raiz,FabricacionSemana,RitmoSemana,CumpFabRit,ExpedidasSemana,Demanda,CumpExpDem,StockFin,StockReal,CumpStock,AcumFab,AcumRitmo,PorcenCump,AcumEnvio,AcumDemanda,PorcenCumpto,EntregasPrev,DemandActual,CumpTotal,customer,uat,destination,lost,delivered
0,45659,6526S,0,-230,0,0,1515,0,910,2816,"309,5",0,0,0,0,0,0,2304,5764,40,010172,UATZ3,010172D01,0,192
1,45659,6695S0020,0,-1,0,0,15,0,64,324,"506,2999878",0,0,0,0,0,0,63,57,"110,5",097946,UATP11,097946D03,0,18
2,45659,6695S0050,0,-51,0,0,295,0,93,270,"290,2999878",0,0,0,0,0,0,1602,1121,"142,8999939",097946,UATM11,097946D03,45,45
3,45659,6695T,0,-280,0,0,1515,0,793,1017,"128,1999969",0,0,0,0,0,0,5229,5757,"90,80000305",097946,UATE5,097946D03,0,378
4,45659,6768S,0,-100,0,0,500,0,400,1208,302,0,0,0,0,0,0,1850,1900,"97,40000153",093929,UATP19,093929D01,0,120


### Dataset Dimensions

In [3]:
print(f'Rows: {df.shape[0]}')
print(f'Columns: {df.shape[1]}')

Rows: 33971
Columns: 25


## 3. Initial Dataset Structure

The dataset structure is inspected to identify the available variables and verify their initial data types.

In [4]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 33971 entries, 0 to 33970
Data columns (total 25 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Fecha              33971 non-null  int64
 1   Grupo_raiz         33971 non-null  str  
 2   FabricacionSemana  33971 non-null  int64
 3   RitmoSemana        33971 non-null  int64
 4   CumpFabRit         33971 non-null  str  
 5   ExpedidasSemana    33971 non-null  int64
 6   Demanda            33971 non-null  int64
 7   CumpExpDem         33971 non-null  str  
 8   StockFin           33971 non-null  int64
 9   StockReal          33971 non-null  int64
 10  CumpStock          33971 non-null  str  
 11  AcumFab            33971 non-null  int64
 12  AcumRitmo          33971 non-null  int64
 13  PorcenCump         33971 non-null  str  
 14  AcumEnvio          33971 non-null  int64
 15  AcumDemanda        33971 non-null  int64
 16  PorcenCumpto       33971 non-null  str  
 17  EntregasPrev       3397

In [5]:
df.columns.tolist()

['Fecha',
 'Grupo_raiz',
 'FabricacionSemana',
 'RitmoSemana',
 'CumpFabRit',
 'ExpedidasSemana',
 'Demanda',
 'CumpExpDem',
 'StockFin',
 'StockReal',
 'CumpStock',
 'AcumFab',
 'AcumRitmo',
 'PorcenCump',
 'AcumEnvio',
 'AcumDemanda',
 'PorcenCumpto',
 'EntregasPrev',
 'DemandActual',
 'CumpTotal',
 'customer',
 'uat',
 'destination',
 'lost',
 'delivered']

## 4. Variable Groups

The dataset contains different types of variables. They are grouped according to their analytical role to simplify subsequent validation and transformation steps.

In [6]:
date_cols = [
    "Fecha"
]

categorical_cols = [
    "Grupo_raiz",
    "customer",
    "uat",
    "destination"
]

operational_absolute_cols = [
    "FabricacionSemana",
    "RitmoSemana",
    "ExpedidasSemana",
    "Demanda",
    "StockFin",
    "StockReal",
    "AcumFab",
    "AcumRitmo",
    "AcumEnvio",
    "AcumDemanda",
    "EntregasPrev",
    "DemandActual"
]

operational_ratio_cols = [
    "CumpFabRit",
    "CumpExpDem",
    "CumpStock",
    "PorcenCump",
    "PorcenCumpto",
    "CumpTotal"
]

outcome_cols = [
    "lost",
    "delivered"
]

numeric_operational_cols = (
    operational_absolute_cols
    + operational_ratio_cols
)

## 5. Variable Description

The dataset combines four main categories of information:

1. **Identification variables**
   - Delivery date and planning group.

2. **Operational absolute variables**
   - Production, demand, stock and accumulated operational quantities.

3. **Operational performance indicators**
   - Ratios describing the relationship between actual and expected operational performance.

4. **Delivery outcome variables**
   - Delivered and lost quantities.

In [7]:
variable_description = pd.DataFrame({
    "Variable": [
        "Fecha",
        "Grupo_raiz",

        "FabricacionSemana",
        "RitmoSemana",
        "CumpFabRit",

        "ExpedidasSemana",
        "Demanda",
        "CumpExpDem",

        "StockFin",
        "StockReal",
        "CumpStock",

        "AcumFab",
        "AcumRitmo",
        "PorcenCump",

        "AcumEnvio",
        "AcumDemanda",
        "PorcenCumpto",

        "EntregasPrev",
        "DemandActual",
        "CumpTotal",

        "customer",
        "uat",
        "destination",

        "lost",
        "delivered"
    ],

    "Category": [
        "Temporal",
        "Identification",

        "Operational absolute",
        "Operational absolute",
        "Operational ratio",

        "Operational absolute",
        "Operational absolute",
        "Operational ratio",

        "Operational absolute",
        "Operational absolute",
        "Operational ratio",

        "Operational absolute",
        "Operational absolute",
        "Operational ratio",

        "Operational absolute",
        "Operational absolute",
        "Operational ratio",

        "Operational absolute",
        "Operational absolute",
        "Operational ratio",

        "Categorical",
        "Categorical",
        "Categorical",

        "Outcome",
        "Outcome"
    ]
})

variable_description

,Variable,Category
0,Fecha,Temporal
1,Grupo_raiz,Identification
2,FabricacionSemana,Operational absolute
3,RitmoSemana,Operational absolute
4,CumpFabRit,Operational ratio
5,ExpedidasSemana,Operational absolute
6,Demanda,Operational absolute
7,CumpExpDem,Operational ratio
8,StockFin,Operational absolute
9,StockReal,Operational absolute


### Operational Performance Indicators

The operational ratios are calculated from their corresponding absolute variables.

#### Weekly Production Compliance

$$
\text{CumpFabRit} =
\frac{\text{FabricacionSemana}}{\text{RitmoSemana}}
\times 100
$$

This indicator measures the percentage of the expected weekly production rhythm that was actually achieved.

#### Weekly Shipment-Demand Compliance

$$
\text{CumpExpDem} =
\frac{\text{ExpedidasSemana}}{\text{Demanda}}
\times 100
$$

This indicator compares weekly shipped quantity with the corresponding demand.

#### Stock Compliance

$$
\text{CumpStock} =
\frac{\text{StockReal}}{\text{StockFin}}
\times 100
$$

This indicator compares actual stock with the expected final stock.

#### Accumulated Production Compliance

$$
\text{PorcenCump} =
\frac{\text{AcumFab}}{\text{AcumRitmo}}
\times 100
$$

#### Accumulated Shipment Compliance

$$
\text{PorcenCumpto} =
\frac{\text{AcumEnvio}}{\text{AcumDemanda}}
\times 100
$$

#### Total Delivery-Demand Compliance

$$
\text{CumpTotal} =
\frac{\text{EntregasPrev}}{\text{DemandActual}}
\times 100
$$

## 6. Data Type Conversion

The raw date is stored using the Microsoft Excel serial date format and must be converted into a standard datetime representation.

In [8]:
df["Fecha"] = pd.to_datetime(
    df["Fecha"],
    unit="D",
    origin="1899-12-30"
)

In [9]:
df["Fecha"].dtype

dtype('<M8[s]')

### Operational Numeric Conversion

Operational variables were initially loaded as strings because decimal values use a comma as decimal separator.

The comma is therefore normalized before converting the variables into numeric format.

In [10]:
for col in numeric_operational_cols:

    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", ".", regex=False)
    )

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

In [11]:
df[numeric_operational_cols].dtypes

FabricacionSemana      int64
RitmoSemana            int64
ExpedidasSemana        int64
Demanda                int64
StockFin               int64
StockReal              int64
AcumFab                int64
AcumRitmo              int64
AcumEnvio              int64
AcumDemanda            int64
EntregasPrev           int64
DemandActual           int64
CumpFabRit           float64
CumpExpDem           float64
CumpStock            float64
PorcenCump           float64
PorcenCumpto         float64
CumpTotal            float64
dtype: object

### Conversion Validation

After converting the operational variables, missing values are checked to ensure that no information was lost because of invalid numeric formats.

In [12]:
conversion_missing = (
    df[numeric_operational_cols]
    .isna()
    .sum()
)

conversion_missing

FabricacionSemana    0
RitmoSemana          0
ExpedidasSemana      0
Demanda              0
StockFin             0
StockReal            0
AcumFab              0
AcumRitmo            0
AcumEnvio            0
AcumDemanda          0
EntregasPrev         0
DemandActual         0
CumpFabRit           0
CumpExpDem           0
CumpStock            0
PorcenCump           0
PorcenCumpto         0
CumpTotal            0
dtype: int64

In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 33971 entries, 0 to 33970
Data columns (total 25 columns):
 #   Column             Non-Null Count  Dtype        
---  ------             --------------  -----        
 0   Fecha              33971 non-null  datetime64[s]
 1   Grupo_raiz         33971 non-null  str          
 2   FabricacionSemana  33971 non-null  int64        
 3   RitmoSemana        33971 non-null  int64        
 4   CumpFabRit         33971 non-null  float64      
 5   ExpedidasSemana    33971 non-null  int64        
 6   Demanda            33971 non-null  int64        
 7   CumpExpDem         33971 non-null  float64      
 8   StockFin           33971 non-null  int64        
 9   StockReal          33971 non-null  int64        
 10  CumpStock          33971 non-null  float64      
 11  AcumFab            33971 non-null  int64        
 12  AcumRitmo          33971 non-null  int64        
 13  PorcenCump         33971 non-null  float64      
 14  AcumEnvio          33971 non-null

## 7. Missing Values

Missing values are analysed across the entire dataset before performing further transformations.

In [14]:
missing_summary = pd.DataFrame({
    "missing_values": df.isna().sum(),
    "missing_percentage": df.isna().mean() * 100
})

missing_summary

,missing_values,missing_percentage
Fecha,0,0.000
Grupo_raiz,0,0.000
FabricacionSemana,0,0.000
RitmoSemana,0,0.000
CumpFabRit,0,0.000
ExpedidasSemana,0,0.000
Demanda,0,0.000
CumpExpDem,0,0.000
StockFin,0,0.000
StockReal,0,0.000


In [15]:
print(
    f"Total missing values: "
    f"{df.isna().sum().sum():,}"
)

Total missing values: 0


## 8. Duplicate Analysis

Duplicate observations can distort descriptive statistics and Machine Learning models.

Both exact duplicates and duplicates according to the expected business granularity are therefore evaluated.

In [16]:
exact_duplicates = df.duplicated().sum()
print(f'Exact duplicates: {exact_duplicates}')

Exact duplicates: 0


In [17]:
business_key = [
    "Fecha",
    "Grupo_raiz"
]

business_duplicates = df.duplicated(
    subset=business_key
).sum()

print(
    f"Duplicated Fecha + Grupo_raiz combinations: "
    f"{business_duplicates:,}"
)

Duplicated Fecha + Grupo_raiz combinations: 0


## 9. Dataset Granularity

The analytical unit of the dataset is defined as:

> **One observation = one planning group on one specific date.**

Delivery information associated with the same planning group and date was aggregated during the SQL extraction stage.

This granularity will also define the unit used during the predictive modelling stage.

In [18]:
granularity_check = (
    df.groupby(
        ["Fecha", "Grupo_raiz"]
    )
    .size()
)

granularity_check.value_counts()

1    33971
Name: count, dtype: int64

## 10. Historical Coverage

The historical period represented in the dataset is analysed before constructing temporal predictive features.

In [19]:
start_date = df["Fecha"].min() 
end_date = df["Fecha"].max()
print(
    f"Start date: {start_date:%Y-%m-%d}\n"
    f"End date: {end_date:%Y-%m-%d}"
)

print(f"Historical duration"
      f": {(end_date - start_date).days:,} days")

Start date: 2025-01-02
End date: 2026-08-25
Historical duration: 600 days


### Dataset Cardinality

The number of unique planning groups, customers, UATs and destinations is analysed to understand the operational scope of the dataset.

In [20]:
cardinality = pd.Series({
    "Planning groups": df["Grupo_raiz"].nunique(),
    "Customers": df["customer"].nunique(),
    "UATs": df["uat"].nunique(),
    "Desinations": df["destination"].nunique()
     }
)
cardinality

Planning groups    226
Customers           55
UATs                21
Desinations         84
dtype: int64

## 11. Historical Coverage by Planning Group

The number of historical observations available for each planning group is analysed.

Planning groups with limited historical coverage may not provide sufficient information for reliable temporal feature engineering.

In [21]:
group_coverage = (
    df.groupby("Grupo_raiz")
    .size()
    .sort_values(ascending=False)
)
group_coverage

Grupo_raiz
B015M         385
7114P0010     384
8850T0040     379
8732P0010     372
B470M001A     370
             ... 
10238S0010      2
8940O           2
A298M-B1        1
A298M-A1        1
C201M0010       1
Length: 226, dtype: int64

In [22]:
group_coverage.describe()

count   226.000
mean    150.314
std     126.396
min       1.000
25%      33.750
50%     113.500
75%     268.250
max     385.000
dtype: float64

In [23]:
group_coverage.head(10)

Grupo_raiz
B015M        385
7114P0010    384
8850T0040    379
8732P0010    372
B470M001A    370
7891S        370
7883P001A    369
7910WA010    368
7890S        366
B946M0010    366
dtype: int64

In [24]:
group_coverage.tail(10)

Grupo_raiz
B0278MA010    5
6526S004A     4
C200M0010     3
8110T         3
8300TA0K0     3
10238S0010    2
8940O         2
A298M-B1      1
A298M-A1      1
C201M0010     1
dtype: int64

## 12. Operational Variable Validation

Operational variables are inspected before being used in statistical analysis or Machine Learning.

The purpose of this section is to understand their scale, dispersion and extreme values without automatically assuming that unusual observations represent errors.

In [25]:
df[
    numeric_operational_cols
].describe().T


,count,mean,std,min,25%,50%,75%,max
FabricacionSemana,"33,971.000","1,502.645","2,792.785","-50,620.000",156.000,555.000,"1,650.000","29,679.000"
RitmoSemana,"33,971.000","1,566.462","2,833.496","-4,800.000",180.000,600.000,"1,700.000","27,000.000"
ExpedidasSemana,"33,971.000","1,472.324","2,690.993",0.000,144.000,540.000,"1,590.000","29,850.000"
Demanda,"33,971.000","1,950.940","3,203.068",0.000,290.000,850.000,"2,070.000","27,645.000"
StockFin,"33,971.000","1,673.632","3,847.405","-10,528.000",224.000,575.000,"1,490.000","63,000.000"
StockReal,"33,971.000","1,727.050","4,139.083",0.000,240.000,611.000,"1,628.000","67,830.000"
AcumFab,"33,971.000","3,534.313","7,320.047","-93,544.000",288.000,"1,102.000","3,609.500","111,784.000"
AcumRitmo,"33,971.000","3,727.080","7,570.842",0.000,328.000,"1,170.000","3,750.000","119,600.000"
AcumEnvio,"33,971.000","3,473.369","7,169.295",0.000,280.000,"1,070.000","3,502.500","100,200.000"
AcumDemanda,"33,971.000","3,636.916","7,425.844",0.000,329.000,"1,143.000","3,660.500","107,824.000"


In [26]:
ratio_percentiles = (
    df[operational_ratio_cols]
    .quantile([
        0.01,
        0.05,
        0.25,
        0.50,
        0.75,
        0.95,
        0.99
    ])
    .T
)

ratio_percentiles

,0.010,0.050,0.250,0.500,0.750,0.950,0.990
CumpFabRit,0.000,0.000,64.700,93.800,115.700,186.900,356.060
CumpExpDem,0.000,0.000,47.100,84.200,102.300,144.400,243.750
CumpStock,-10.760,20.000,63.100,105.300,166.300,346.200,728.600
PorcenCump,0.000,0.000,73.600,94.400,109.800,162.400,292.060
PorcenCumpto,0.000,0.000,75.000,96.100,107.100,151.050,254.860
CumpTotal,30.000,67.700,93.900,100.000,105.000,140.000,219.580


### Negative Operational Values

Some operational quantities contain negative observations.

According to the operational context, these values may represent adjustments, corrections or carry-over effects from previous planning periods.

They are therefore preserved instead of being automatically removed as invalid observations.

In [27]:
negative_summary = pd.DataFrame({
    "negative_count":
        (df[operational_absolute_cols] < 0).sum(),

    "negative_percentage":
        (
            df[operational_absolute_cols] < 0
        ).mean() * 100
})

negative_summary

,negative_count,negative_percentage
FabricacionSemana,93,0.274
RitmoSemana,628,1.849
ExpedidasSemana,0,0.000
Demanda,0,0.000
StockFin,389,1.145
StockReal,0,0.000
AcumFab,90,0.265
AcumRitmo,0,0.000
AcumEnvio,0,0.000
AcumDemanda,0,0.000


### Zero Denominators

Several operational performance indicators depend on quantities that may occasionally be zero.

Zero or very small denominators can produce undefined or very large percentage values.

These observations are therefore analysed explicitly rather than treating extreme ratios automatically as outliers.

In [28]:
denominator_cols = [
    "RitmoSemana",
    "Demanda",
    "StockFin",
    "AcumRitmo",
    "AcumDemanda",
    "DemandActual"
]

In [29]:
zero_denominators = pd.DataFrame({
    "zero_count":
        (df[denominator_cols] == 0).sum(),

    "zero_percentage":
        (
            df[denominator_cols] == 0
        ).mean() * 100
})

zero_denominators

,zero_count,zero_percentage
RitmoSemana,1850,5.446
Demanda,110,0.324
StockFin,83,0.244
AcumRitmo,2200,6.476
AcumDemanda,2121,6.244
DemandActual,110,0.324


## 13. Delivery Quantity Validation

Delivered and lost quantities represent the final logistics outcome of each planning-group/date observation.

These variables are validated before constructing the prediction target.

In [30]:
df[
    ["lost", "delivered"]
].describe()

,lost,delivered
count,"33,971.000","33,971.000"
mean,40.700,503.061
std,276.302,868.041
min,0.000,0.000
25%,0.000,60.000
50%,0.000,192.000
75%,0.000,578.000
max,"12,600.000","13,800.000"


In [31]:
print(
    "Negative lost quantities:",
    (df["lost"] < 0).sum()
)

print(
    "Negative delivered quantities:",
    (df["delivered"] < 0).sum()
)

Negative lost quantities: 0
Negative delivered quantities: 0


### Requested Quantity

The required quantity associated with each observation is reconstructed as:

\[
Requested = Delivered + Lost
\]

This represents the total quantity that should have been successfully delivered.

In [32]:
df["requested"] = df["lost"] + df["delivered"] 

In [33]:
df[
    ["delivered", "lost", "requested"]
].describe()

,delivered,lost,requested
count,"33,971.000","33,971.000","33,971.000"
mean,503.061,40.700,543.761
std,868.041,276.302,927.921
min,0.000,0.000,1.000
25%,60.000,0.000,70.000
50%,192.000,0.000,216.000
75%,578.000,0.000,630.000
max,"13,800.000","12,600.000","23,640.000"


## 14. Missed Delivery Target Definition

The predictive problem is formulated as a binary classification task.

A planning-group/date observation is classified as a Missed Delivery when at least one required element was not successfully delivered.

\[
MissedDelivery =
\begin{cases}
1, & lost > 0 \\
0, & lost = 0
\end{cases}
\]

In [34]:
df["missed_delivery"] = (
    df["lost"] > 0
).astype(int)

In [35]:
target_summary = (
    df["missed_delivery"]
    .value_counts()
    .sort_index()
    .to_frame("records")
)

target_summary["percentage"] = (
    target_summary["records"]
    / len(df)
    * 100
)

target_summary.index = [
    "No Missed Delivery",
    "Missed Delivery"
]

target_summary

,records,percentage
No Missed Delivery,31256,92.008
Missed Delivery,2715,7.992


In [36]:
global_md_rate = (
    df["missed_delivery"].mean()
    * 100
)

print(
    f"Global Missed Delivery Rate: "
    f"{global_md_rate:.2f}%"
)

Global Missed Delivery Rate: 7.99%


## 15. Missed Delivery Frequency and Volume Impact

Missed Delivery frequency alone does not fully represent business impact.

A relatively small number of high-volume failures may generate a greater operational impact than a large number of low-volume failures.

For this reason, two complementary perspectives are considered:

### Frequency

Percentage of planning-group/date observations containing at least one lost element.

### Volume Impact

Percentage of requested pieces that were not successfully delivered.

In [37]:
total_requested = df["requested"].sum()
total_delivered = df["delivered"].sum()
total_lost = df["lost"].sum()

quantity_miss_rate = (
    total_lost
    / total_requested 
    * 100
)
print(
    f"Requested pieces: {total_requested:,.0f}"
)

print(
    f"Delivered pieces: {total_delivered:,.0f}"
)

print(
    f"Lost pieces:      {total_lost:,.0f}"
)

print(
    f"Quantity Miss Rate: "
    f"{quantity_miss_rate:.2f}%"
)

Requested pieces: 18,472,090
Delivered pieces: 17,089,477
Lost pieces:      1,382,613
Quantity Miss Rate: 7.48%


## 16. Initial Missed Delivery Severity

The amount of lost quantity associated with Missed Delivery observations is inspected to obtain an initial measure of failure severity.

Detailed high-volume Missed Delivery analysis will be performed during the Exploratory Data Analysis stage.

In [38]:
missed_df = df[
    df["missed_delivery"] == 1
].copy()

missed_df["lost"].describe()

count    2,715.000
mean       509.250
std        846.672
min          1.000
25%         72.000
50%        208.000
75%        576.000
max     12,600.000
Name: lost, dtype: float64

In [39]:
missed_df["lost"].quantile(
    [
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ]
)

0.500     208.000
0.750     576.000
0.900   1,341.200
0.950   1,980.000
0.990   4,095.000
Name: lost, dtype: float64

In [40]:
dataset_summary = pd.Series({
    "Historical start":
        df["Fecha"].min().date(),

    "Historical end":
        df["Fecha"].max().date(),

    "Records":
        len(df),

    "Planning groups":
        df["Grupo_raiz"].nunique(),

    "Customers":
        df["customer"].nunique(),

    "UATs":
        df["uat"].nunique(),

    "Destinations":
        df["destination"].nunique(),

    "Missed Delivery records":
        df["missed_delivery"].sum(),

    "Missed Delivery Rate (%)":
        round(
            df["missed_delivery"].mean()
            * 100,
            2
        ),

    "Requested pieces":
        df["requested"].sum(),

    "Delivered pieces":
        df["delivered"].sum(),

    "Lost pieces":
        df["lost"].sum(),

    "Quantity Miss Rate (%)":
        round(
            df["lost"].sum()
            / df["requested"].sum()
            * 100,
            2
        )
})

dataset_summary

Historical start            2025-01-02
Historical end              2026-08-25
Records                          33971
Planning groups                    226
Customers                           55
UATs                                21
Destinations                        84
Missed Delivery records           2715
Missed Delivery Rate (%)         7.990
Requested pieces              18472090
Delivered pieces              17089477
Lost pieces                    1382613
Quantity Miss Rate (%)           7.480
dtype: object

## 17. Initial Conclusions

The initial Data Understanding stage confirms that the dataset provides an analytical representation of delivery performance at planning-group and date level.

The available information combines absolute operational measurements, performance indicators and final delivery outcomes.

The predictive problem can be formulated as a binary classification task, where each planning-group/date observation is classified according to whether a Missed Delivery occurred.

The frequency of Missed Deliveries and the corresponding lost volume are treated as complementary business perspectives. This distinction is particularly relevant because high-volume failures may have greater operational impact even when their frequency is relatively low.

Negative operational quantities are preserved because they may represent valid adjustments or carry-over effects from previous planning periods.

Similarly, extreme percentage values are not automatically removed, since they may be produced by low or zero denominators and may contain relevant operational information.

The processed dataset is considered suitable for subsequent Exploratory Data Analysis, statistical analysis and temporal feature engineering.

### Planning Group Historical Coverage

The dataset contains 226 unique planning groups with heterogeneous historical coverage.

The median planning group contains approximately 114 observations, while 25% of the groups contain fewer than approximately 34 observations. Some groups contain only a single historical observation.

This variability in historical coverage must be considered during temporal feature engineering, since groups with insufficient history may not support reliable lag-based features or predictive modelling.

### Operational Ratio Distribution

The operational compliance indicators present highly right-skewed distributions.

Although most observations are concentrated around operationally reasonable values, a reduced number of observations reach extremely high values. For example, the median `CumpStock` value is approximately 105%, while its 99th percentile exceeds 700%.

These observations are not automatically removed because extreme ratios may result from very small denominators or valid operational conditions.

Robust transformation and scaling strategies will be evaluated during the Machine Learning preprocessing stage.

### Negative Operational Values

Negative values represent a small proportion of the dataset.

The highest proportion occurs in `RitmoSemana`, where approximately 1.85% of observations are negative. Negative values are also present in `StockFin`, `FabricacionSemana` and `AcumFab`.

According to the operational context, these values may represent adjustments or carry-over effects from previous planning periods. Therefore, they are preserved for subsequent analysis.

The dataset contains 2,715 Missed Delivery observations, corresponding to approximately 7.99% of all planning-group/date observations.

From a volume perspective, approximately 1.38 million pieces were lost out of 18.47 million requested pieces, resulting in a Quantity Miss Rate of approximately 7.48%.

The similarity between the event-based and volume-based failure rates indicates that Missed Deliveries represent not only a frequency issue but also a relevant volumetric business impact.

## 18. Export Processed Dataset

After validating data types, granularity, operational variables and delivery outcomes, the processed dataset is exported for subsequent notebooks.

Temporal lag variables are intentionally not generated during this stage because they belong to the Feature Engineering phase.

In [41]:
output_path = "../../data/processed/dataset_clean.csv"

df.to_csv(
    output_path,
    index=False
)

print(
    f"Processed dataset exported to: "
    f"{output_path}"
)

print(
    f"Final Shape"
    f"{df.shape[0]} rows x"
    f"{df.shape[1]} columns"
)

Processed dataset exported to: ../../data/processed/dataset_clean.csv
Final Shape33971 rows x27 columns
